# Validate a prepared physical realization

The preparation layer performs HEALPix geometry, seeded random-field synthesis, and shell-catalog construction once. The resulting `SkyInputs` then enter the pure JAX model. The deliberately low resolution keeps this validation suitable for a laptop.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from lowsky import SkyConfig, SkyParameters, generate_sky, prepare_sky_inputs

In [ ]:
config = SkyConfig(
    nside=2, ray_oversample=1, n_distance=12, max_distance_kpc=12.0,
    n_shells=2, shell_distance_bins=4, shell_quadrature_steps=16, seed=7,
)
inputs = prepare_sky_inputs(config)
frequencies = jnp.array([3.0, 10.0, 30.0, 50.0])
parameters = SkyParameters()
sky = jax.jit(generate_sky)(frequencies, inputs, parameters)
assert sky.shape == (4, 12 * config.nside**2)
assert jnp.all(jnp.isfinite(sky)) and jnp.all(sky >= 0.0)

In [ ]:
beta_gradient = jax.grad(
    lambda offset: jnp.mean(generate_sky(
        frequencies, inputs, parameters._replace(spectral_index_offset=offset)
    ))
)(jnp.asarray(0.0))
assert jnp.isfinite(beta_gradient)
print(f"mean-temperature gradient with spectral-index offset = {float(beta_gradient):.3e}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.5))
median = jnp.median(sky, axis=1)
low, high = jnp.percentile(sky, jnp.array([10.0, 90.0]), axis=1)
ax.fill_between(frequencies, low, high, alpha=0.25, label="10th–90th percentile")
ax.loglog(frequencies, median, marker="o", label="median")
ax.set(xlabel="Frequency [MHz]", ylabel="Temperature [K]", title="Prepared physical sky")
ax.grid(True, which="both", alpha=0.25)
ax.legend();